# 03 · Order-book price impact: rolling Kyle's λ from order-flow imbalance

Kyle (1985): $\Delta P_t = \alpha + \lambda S_t + \varepsilon_t$. With level-1 order-book snapshots instead of a
trade tape, signed flow $S_t$ is proxied by the Cont–Kukanov–Stoikov (2014) order-flow imbalance

$$e_t = \mathbb 1\{P^b_t \ge P^b_{t-1}\}q^b_t - \mathbb 1\{P^b_t \le P^b_{t-1}\}q^b_{t-1}
      - \mathbb 1\{P^a_t \le P^a_{t-1}\}q^a_t + \mathbb 1\{P^a_t \ge P^a_{t-1}\}q^a_{t-1}.$$

Snapshots are summed into one-minute bars and λ is re-estimated by OLS on a rolling 1,440-minute window.

> **Data.** CoinAPI order books are licensed and are **not** in this repository. Place them under
> `data/external/orderbook/<YYYY-MM-DD>/<PAIR>_ob_<YYYY-MM-DD>.csv`. The paper's Figure 3 uses Coinbase BTC/USD and
> BTC/USDT; this run uses the Binance BTC/USDT book, the pair with full 21-day coverage locally.

In [1]:
# Make the package importable without `pip install -e .` (harmless if already installed)
import sys, pathlib
here = pathlib.Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(root / "src"))

import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api  # noqa: F401  (statsmodels re-enables its warnings on import, so silence afterwards)
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
%matplotlib inline

In [2]:
from stablecoin_risk import config, microstructure as ms, plotting
from stablecoin_risk.data import loaders

root = config.EXTERNAL_DIR / "orderbook"
if not root.exists():
    raise SystemExit("CoinAPI order books not available - see the note above.")
ob = loaders.load_orderbook("BINANCE_BTC_USDT", root)
print(f"{len(ob):,} snapshots, {ob.index.normalize().nunique()} days")
ob.head()

27,283 snapshots, 21 days


,ask_price_1,ask_size_1,bid_price_1,bid_size_1,mid_price
time_exchange,,,,,
2023-03-01 00:00:00.010000+00:00,2.314e+04,0.02242,2.314e+04,0.0019,2.314e+04
2023-03-01 00:01:00.637000+00:00,2.314e+04,0.00139,2.314e+04,0.00118,2.314e+04
2023-03-01 00:02:00.265000+00:00,2.314e+04,0.00185,2.314e+04,0.0007,2.314e+04
2023-03-01 00:03:00.995000+00:00,2.316e+04,0.1964,2.316e+04,0.001,2.316e+04
2023-03-01 00:04:00.023000+00:00,2.316e+04,0.07489,2.316e+04,0.00047,2.316e+04


In [3]:
lam = ms.kyle_lambda_ofi(ob)
pd.Series({r: lam.loc[a:f"{b} 23:59"].median() for r, (a, b) in config.REGIMES.items()}, name="median lambda")

pre_crisis    17.13
peak_crisis   8.975
recovery      19.71
Name: median lambda, dtype: float64

In [4]:
fig = plotting.plot_svb_panels({"Rolling Kyle's lambda, Binance BTC/USDT": {"BINANCE_BTC_USDT": lam}},
                               "price change per unit OFI", resample="30min")

On Binance BTC/USDT, price impact per unit of order flow is *lower* through the crisis window than in the
surrounding weeks: the USDT book thickened as flow migrated out of USDC pairs. This is the same "USDT as refuge"
pattern the paper documents on Coinbase BTC/USDT.